In [1]:
import numpy as np
from scipy.spatial.distance import cdist
class Nystrom():
    def __init__(self,loss,n,m,lambda_,alpha,gamma,iterations):
        self.n = n
        self.m = m
        self.lambda_ = lambda_
        self.alpha = alpha
        self.gamma = gamma
        self.iterations = iterations
        self.loss = loss
    
    def kernel(self,X, Y=None, length_scale=1.0, variance=1.0):
        X = np.asarray(X, dtype=float).reshape(-1, 1)  # shape (n,1)
        if Y is None:
            Y = X
        else:
            Y = np.asarray(Y, dtype=float).reshape(-1, 1)  # shape (1,m)
        d = np.abs(X - Y.T)  # broadcasting (n,1) - (1,m) -> (n,m)
        sqrt5_d = np.sqrt(5) * d / length_scale
        K = variance * (1 + sqrt5_d + (5/3) * (d**2) / length_scale**2) * np.exp(-sqrt5_d)
        return K
    
    def leverage_score(self):
        n = self.n
        if 3*self.m > n:
            s = n
        else:
            s = 3*self.m
        X_s = np.random.choice(self.X, s, replace=False)
        K_ns = self.kernel(self.X, X_s)
        K_ss = self.kernel(X_s)
        eigen, U = np.linalg.eigh(K_ss + 10**(-8)*np.eye(s))
        eigen = np.maximum(eigen, 10**(-8))
        D_inv_sqrt = np.diag(1.0 / np.sqrt(eigen))
        K_ss_inv_sqrt = np.dot(np.dot(U, D_inv_sqrt), U.T)
        Z = np.dot(K_ns, K_ss_inv_sqrt)
        M = np.dot(Z.T, Z) + self.alpha * n * np.eye(s)
        M_inv = np.linalg.inv(M)
        hat_l = np.sum(np.dot(Z, M_inv) * Z, axis=1)
        hat_l = np.maximum(hat_l, 0.0)
        total = hat_l.sum()
        if total <= 0:
            p = np.ones(n) / n
        else:
            p = hat_l / total
        nystrom_idx = np.random.choice(n, size=self.m, replace=False, p=p)
        X_m = self.X[nystrom_idx]
        return X_m
    
    def derivative_loss(self,resid):
        if self.loss == 'Cauchy':
            derivative = resid/(1+0.5*resid**2)
        elif self.loss == 'Huber':
            derivative = resid/np.sqrt(resid**2+1.)
        elif self.loss == 'Welsch':
            derivative = resid * np.exp(-0.5 * resid**2) 
        return derivative
        
    def iteration(self):
        self.X,self.Y,opq = self.construct_data(self.n)
        self.X_m = self.leverage_score()
        K_nm = self.kernel(self.X, self.X_m)
        K_mm = self.kernel(self.X_m)
        eigen, U = np.linalg.eigh(K_mm + 10**(-8) * np.eye(self.m))
        eigen = np.maximum(eigen, 10**(-8))
        D_inv_sqrt = np.diag(1.0 / np.sqrt(eigen))
        K_mm_inv_sqrt = np.dot(np.dot(U, D_inv_sqrt), U.T)
        Z = np.dot(K_nm, K_mm_inv_sqrt)
        a = np.zeros(self.m)
        n = len(self.X)
        for i in range(self.iterations):
            pred = np.dot(Z, a)
            grad_u = self.derivative_loss(pred-self.Y)
            grad = np.dot(Z.T, grad_u) / n + 2.0 * self.lambda_ * a
            a = a - self.gamma*grad
        self.coe = np.dot(K_mm_inv_sqrt, a)
        return self.coe

    def grho_cal(self,theta):
        t = theta/(2*math.pi)
        return (t**4-2*(t**3)+t**2-1/30)/2
        
    def construct_data(self,N):
        X = np.random.uniform(0, 1, size=N)
        Y = (X**2-X+1/6)/5
        noise = np.random.uniform(-0.2, 0.2, size=N)
        Z = Y + noise
        return [X,Z,Y]
        
    def test(self):
        X,Z,Y = self.construct_data(1000)
        K_test = self.kernel(X, self.X_m)
        predict = np.dot(K_test, self.coe)
        return np.mean((predict - Y)**2)
        
    

In [ ]:
import time
Loss = ['Cauchy', 'Huber', 'Welsch']
N = [int(10**(i/5)) for i in range(15,26)]
T = 20
for loss in  Loss:
    err = []
    err_ave = []
    time_list = []
    time_ave = []
    for i in range(T):
        E = []
        Time = []
        for n in N:
            m = int(n**(1/2))+50
            lambda_ = 10**(-3)/np.sqrt(n)
            alpha = 30/n
            gamma = 0.5
            iterations = 1000
            start = time.perf_counter()
            nystrom = Nystrom(loss,n,m,lambda_,alpha,gamma,iterations)
            coe = nystrom.iteration()
            end = time.perf_counter()
            t = end-start
            E.append(nystrom.test())
            Time.append(t)
            #print(n,E[-1],t)
        err.append(E[:])
        time_list.append(Time[:])
    for i in range(len(N)):
        S = 0
        t_time = 0
        for j in range(T):
            S += err[j][i]
            t_time += time_list[j][i]
        err_ave.append(S/T)
        time_ave.append(t_time/T)
    print('nystrom_'+loss+'_B2 =',err_ave)
    print(' ')
    print('time_nystrom_'+loss+'_B2 =',time_ave)